# RAG: PDF Question Answering

**Category:** 04-RAG-Documents  
**Description:** Build a document Q&A system using PDFs from ~/lab/data/PDFs/  
**Data:** Technical papers, books, and documentation

---

## What is RAG?

**Retrieval-Augmented Generation (RAG)** combines:
1. **Retrieval:** Find relevant documents/passages
2. **Augmentation:** Add context to the prompt
3. **Generation:** AI generates answers grounded in documents

This enables AI to answer questions about YOUR documents accurately.

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pymupdf langchain langchain-community chromadb sentence-transformers

# Model aliases - update when models change
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
CLAUDE_FAST = "anthropic-chat:claude-haiku-4-5-20251001"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE  # Default model for this notebook

In [ ]:
import fitz  # PyMuPDF
import os
from pathlib import Path
from typing import List, Dict
import re

PDF_DIR = Path.home() / 'lab' / 'data' / 'PDFs'
DB_DIR = Path.home() / 'lab' / 'data' / 'DBs'

# List available PDFs
pdfs = list(PDF_DIR.glob('*.pdf'))
print(f"Found {len(pdfs)} PDF files:")
for pdf in pdfs:
    size_mb = pdf.stat().st_size / (1024 * 1024)
    print(f"  - {pdf.name} ({size_mb:.1f} MB)")

---

# Part 1: PDF Text Extraction

---

In [ ]:
def extract_pdf_text(pdf_path: Path) -> List[Dict]:
    """
    Extract text from PDF with page-level metadata.
    
    Returns:
        List of dicts with 'text', 'page', 'source' keys
    """
    documents = []
    
    try:
        doc = fitz.open(pdf_path)
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text("text")
            
            # Clean up text
            text = re.sub(r'\s+', ' ', text).strip()
            
            if len(text) > 50:  # Skip nearly empty pages
                documents.append({
                    'text': text,
                    'page': page_num + 1,
                    'source': pdf_path.name,
                    'total_pages': len(doc)
                })
        
        doc.close()
        
    except Exception as e:
        print(f"Error processing {pdf_path.name}: {e}")
    
    return documents

# Test extraction
test_pdf = PDF_DIR / 'incident_metrics_in_sre.pdf'
if test_pdf.exists():
    docs = extract_pdf_text(test_pdf)
    print(f"Extracted {len(docs)} pages from {test_pdf.name}")
    print(f"\nSample from page 1 (first 500 chars):")
    print(docs[0]['text'][:500] if docs else "No content")

## 1.1 Batch Process All PDFs

In [ ]:
def process_all_pdfs(pdf_dir: Path) -> List[Dict]:
    """Process all PDFs in directory."""
    all_docs = []
    
    for pdf_path in pdf_dir.glob('*.pdf'):
        print(f"Processing: {pdf_path.name}")
        docs = extract_pdf_text(pdf_path)
        all_docs.extend(docs)
        print(f"  -> Extracted {len(docs)} pages")
    
    return all_docs

# Process all PDFs (this may take a minute)
print("Processing all PDFs...\n")
all_documents = process_all_pdfs(PDF_DIR)

print(f"\n{'='*50}")
print(f"Total documents extracted: {len(all_documents)}")
print(f"Total text length: {sum(len(d['text']) for d in all_documents):,} characters")

---

# Part 2: Text Chunking

---

For effective retrieval, we split documents into overlapping chunks.

In [ ]:
def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
    """
    Split text into overlapping chunks.
    
    Args:
        text: Input text
        chunk_size: Target size of each chunk
        overlap: Overlap between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        # Try to break at sentence boundary
        if end < len(text):
            # Look for period, question mark, or exclamation
            boundary = text.rfind('.', start + chunk_size - 100, end)
            if boundary > start:
                end = boundary + 1
        
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        
        start = end - overlap
    
    return chunks

# Chunk all documents
chunked_documents = []

for doc in all_documents:
    chunks = chunk_text(doc['text'])
    
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            'text': chunk,
            'source': doc['source'],
            'page': doc['page'],
            'chunk_id': i,
            'total_chunks': len(chunks)
        })

print(f"Created {len(chunked_documents)} chunks from {len(all_documents)} pages")
print(f"Average chunk length: {sum(len(d['text']) for d in chunked_documents) / len(chunked_documents):.0f} chars")

---

# Part 3: Vector Embeddings & Storage

---

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and effective
print("Model loaded!")

In [ ]:
# Initialize ChromaDB
chroma_path = DB_DIR / 'chroma_db'
chroma_path.mkdir(exist_ok=True)

client = chromadb.PersistentClient(path=str(chroma_path))

# Create or get collection
collection = client.get_or_create_collection(
    name="pdf_documents",
    metadata={"description": "PDF documents from lab data"}
)

print(f"Collection initialized: {collection.name}")
print(f"Current document count: {collection.count()}")

In [ ]:
# Add documents to vector store (if not already added)
if collection.count() < len(chunked_documents):
    print(f"Adding {len(chunked_documents)} documents to vector store...")
    
    # Process in batches
    batch_size = 100
    
    for i in range(0, len(chunked_documents), batch_size):
        batch = chunked_documents[i:i+batch_size]
        
        texts = [d['text'] for d in batch]
        embeddings = embedder.encode(texts).tolist()
        
        ids = [f"doc_{i+j}" for j in range(len(batch))]
        metadatas = [{
            'source': d['source'],
            'page': d['page'],
            'chunk_id': d['chunk_id']
        } for d in batch]
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        print(f"  Added batch {i//batch_size + 1}/{(len(chunked_documents)-1)//batch_size + 1}")
    
    print(f"\nTotal documents in collection: {collection.count()}")
else:
    print(f"Collection already has {collection.count()} documents")

---

# Part 4: Semantic Search

---

In [ ]:
def semantic_search(query: str, n_results: int = 5) -> List[Dict]:
    """
    Search for relevant documents using semantic similarity.
    
    Args:
        query: Search query
        n_results: Number of results to return
    
    Returns:
        List of relevant document chunks with metadata
    """
    # Embed query
    query_embedding = embedder.encode([query]).tolist()
    
    # Search
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Format results
    formatted = []
    for i in range(len(results['documents'][0])):
        formatted.append({
            'text': results['documents'][0][i],
            'source': results['metadatas'][0][i]['source'],
            'page': results['metadatas'][0][i]['page'],
            'distance': results['distances'][0][i]
        })
    
    return formatted

In [ ]:
# Test semantic search
query = "How should incidents be measured and tracked?"

print(f"Query: {query}\n")
print("="*60)

results = semantic_search(query, n_results=3)

for i, result in enumerate(results, 1):
    print(f"\n📄 Result {i} (Distance: {result['distance']:.4f})")
    print(f"   Source: {result['source']}, Page {result['page']}")
    print(f"   Preview: {result['text'][:300]}...")

---

# Part 5: Question Answering with RAG

---

In [ ]:
def build_rag_prompt(question: str, context_docs: List[Dict]) -> str:
    """
    Build a RAG prompt with retrieved context.
    """
    context_text = "\n\n".join([
        f"[Source: {d['source']}, Page {d['page']}]\n{d['text']}"
        for d in context_docs
    ])
    
    prompt = f"""Answer the question based ONLY on the following context from technical documents.
If the answer is not in the context, say "I don't have enough information to answer this."
Always cite the source document and page number.

CONTEXT:
{context_text}

QUESTION: {question}

ANSWER:"""
    
    return prompt

In [ ]:
# Example 1: SRE Metrics Question
question = "What are the key metrics used to measure incident response effectiveness?"

# Retrieve relevant context
context = semantic_search(question, n_results=4)

# Build prompt
prompt = build_rag_prompt(question, context)

print("Question:", question)
print("\nRetrieved", len(context), "relevant passages")
print("\n" + "="*60)
print("Sending to AI...")

In [ ]:
%%ai $MODEL
Answer the question based ONLY on the following context from technical documents.
If the answer is not in the context, say "I don't have enough information to answer this."
Always cite the source document and page number.

QUESTION: What are the key metrics used to measure incident response effectiveness?

Based on the context retrieved from the incident_metrics_in_sre.pdf document, provide a comprehensive answer.

## 5.1 More Example Queries

In [ ]:
# Example 2: Google Borg Question
question2 = "How does Google Borg handle resource scheduling?"

context2 = semantic_search(question2, n_results=4)
print(f"Query: {question2}")
print(f"\nTop sources: {set(d['source'] for d in context2)}")
for doc in context2:
    print(f"  - {doc['source']} p.{doc['page']} (distance: {doc['distance']:.3f})")

In [ ]:
%%ai $MODEL
Based on technical documentation about Google Borg, explain how Borg handles resource scheduling and allocation for containerized workloads. Include details about the scheduling algorithm if available.

In [ ]:
# Example 3: Observability Question
question3 = "What is the difference between monitoring and observability?"

context3 = semantic_search(question3, n_results=4)
print(f"Query: {question3}")
print(f"\nTop sources: {set(d['source'] for d in context3)}")

In [ ]:
%%ai $MODEL
Explain the difference between monitoring and observability in modern software systems. Draw from the O'Reilly Observability Engineering book if context is available.

In [ ]:
# Example 4: Control Systems Question
question4 = "What are the fundamental principles of feedback control systems?"

context4 = semantic_search(question4, n_results=4)
print(f"Query: {question4}")
print(f"\nTop sources: {set(d['source'] for d in context4)}")

In [ ]:
%%ai $MODEL
Explain the fundamental principles of feedback control systems, including concepts like stability, response time, and error correction.

In [ ]:
# Example 5: Machine Learning Question
question5 = "What are effective techniques for regularization in machine learning?"

context5 = semantic_search(question5, n_results=4)
print(f"Query: {question5}")
print(f"\nTop sources: {set(d['source'] for d in context5)}")

In [ ]:
%%ai $MODEL
What are the most effective regularization techniques in machine learning to prevent overfitting? Explain L1, L2, and any other techniques with their trade-offs.

---

# Part 6: Interactive Q&A Function

---

In [ ]:
def ask_documents(question: str, n_context: int = 4, verbose: bool = True) -> str:
    """
    Ask a question about the indexed documents.
    
    Args:
        question: Your question
        n_context: Number of context passages to retrieve
        verbose: Print retrieval details
    
    Returns:
        The RAG prompt ready for AI
    """
    # Retrieve context
    context = semantic_search(question, n_results=n_context)
    
    if verbose:
        print(f"📚 Retrieved {len(context)} passages:")
        for i, doc in enumerate(context, 1):
            print(f"   {i}. {doc['source']} (p.{doc['page']}) - distance: {doc['distance']:.3f}")
        print()
    
    # Build prompt
    prompt = build_rag_prompt(question, context)
    
    return prompt

# Example usage
my_question = "How can organizations improve their incident management process?"
rag_prompt = ask_documents(my_question)

In [ ]:
%%ai $MODEL
Based on the retrieved documentation about SRE practices and incident management, 
how can organizations improve their incident management process? 
Provide specific, actionable recommendations.

---

# Part 7: Advanced RAG Techniques

---

## 7.1 Hybrid Search (Semantic + Keyword)

In [ ]:
def hybrid_search(query: str, n_results: int = 5, keyword_boost: float = 0.3) -> List[Dict]:
    """
    Combine semantic search with keyword matching for better results.
    """
    # Get semantic results
    semantic_results = semantic_search(query, n_results=n_results * 2)
    
    # Extract keywords from query
    keywords = set(query.lower().split())
    keywords -= {'what', 'how', 'why', 'when', 'where', 'is', 'are', 'the', 'a', 'an', 'in', 'on', 'for'}
    
    # Score results
    for result in semantic_results:
        text_lower = result['text'].lower()
        keyword_matches = sum(1 for kw in keywords if kw in text_lower)
        keyword_score = keyword_matches / max(len(keywords), 1)
        
        # Combine scores (lower distance is better, higher keyword score is better)
        result['hybrid_score'] = (1 - result['distance']) + (keyword_boost * keyword_score)
    
    # Sort by hybrid score
    semantic_results.sort(key=lambda x: x['hybrid_score'], reverse=True)
    
    return semantic_results[:n_results]

# Test hybrid search
query = "incident metrics MTTR MTTD"
results = hybrid_search(query)

print(f"Hybrid search for: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['source']} p.{r['page']} (score: {r['hybrid_score']:.3f})")

## 7.2 Document-Specific Search

In [ ]:
def search_document(query: str, source_filter: str, n_results: int = 5) -> List[Dict]:
    """
    Search within a specific document.
    """
    query_embedding = embedder.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where={"source": source_filter},
        include=['documents', 'metadatas', 'distances']
    )
    
    formatted = []
    if results['documents'][0]:
        for i in range(len(results['documents'][0])):
            formatted.append({
                'text': results['documents'][0][i],
                'source': results['metadatas'][0][i]['source'],
                'page': results['metadatas'][0][i]['page'],
                'distance': results['distances'][0][i]
            })
    
    return formatted

# Search only in the observability book
obs_results = search_document(
    "distributed tracing",
    "Honeycomb-OReilly-Book-on-Observability-Engineering.pdf"
)

print(f"Found {len(obs_results)} results in Observability Engineering book:")
for r in obs_results:
    print(f"  Page {r['page']}: {r['text'][:100]}...")

---

## Summary

This notebook demonstrated a complete RAG pipeline:

1. **PDF Extraction** - Text extraction with PyMuPDF
2. **Text Chunking** - Overlapping chunks for context preservation
3. **Vector Embeddings** - Sentence transformers for semantic encoding
4. **Vector Storage** - ChromaDB for persistent storage
5. **Semantic Search** - Find relevant passages by meaning
6. **Question Answering** - AI-powered answers with citations
7. **Advanced Techniques** - Hybrid search and document filtering

---

**Next:** Try semantic search over structured data in `04-RAG-Documents/02-structured-data-rag.ipynb`